## 1. Imports and Paths ##

In [197]:
import ast
import pandas as pd
import numpy as np

In [198]:
DATA_PATH = "../Processed/scoring_ready_germany_sample.csv"

LABEL_MAPPING_PATH = "../Data/open_products_labe_mapping_final.csv"
CONFIDENCE_MAPPING_PATH = "../Data/open_products_confidence_mapping_final.csv"

## 2. Load Data ## 

In [199]:
products_df = pd.read_csv(DATA_PATH)

def read_mapping_csv(path):
    return pd.read_csv(path, sep=None, engine="python", encoding="utf-8-sig")

label_mapping = read_mapping_csv(LABEL_MAPPING_PATH)
confidence_mapping = read_mapping_csv(CONFIDENCE_MAPPING_PATH)

label_mapping.columns = label_mapping.columns.str.strip()
confidence_mapping.columns = confidence_mapping.columns.str.strip()

In [200]:
# Clean empty mapping rows

required_label_columns = ["database", "label_keyword", "matched_label", "score_group"]
required_confidence_columns = ["label_keyword", "matched_label", "score_group"]

missing_label_columns = [col for col in required_label_columns if col not in label_mapping.columns]
missing_confidence_columns = [col for col in required_confidence_columns if col not in confidence_mapping.columns]

if missing_label_columns:
    raise ValueError(f"Missing columns in label mapping: {missing_label_columns}. Available columns: {label_mapping.columns.tolist()}")

if missing_confidence_columns:
    raise ValueError(f"Missing columns in confidence mapping: {missing_confidence_columns}. Available columns: {confidence_mapping.columns.tolist()}")

label_mapping = label_mapping.dropna(subset=required_label_columns).copy()
confidence_mapping = confidence_mapping.dropna(subset=required_confidence_columns).copy()

## 3. Source Mapping ##

The Product dataset uses:

- food
- beauty
- products

The mapping file uses:

- open_food_facts
- open_beauty_facts
- open_products_facts
- shared

We will create a converter for this.

In [201]:
SOURCE_TO_DATABASE = {
    "food": "open_food_facts",
    "beauty": "open_beauty_facts",
    "products": "open_products_facts"
}

## 4. Parse List Columns ##

In [202]:
def parse_list_column(value):
    if isinstance(value, list):
        return value

    if pd.isna(value) or value == "":
        return []

    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
        return []
    except Exception:
        return []

In [203]:
# Apply to useful tag columns

list_columns = [
    "label_tags",
    "category_tags",
    "ingredient_tags",
    "packaging_tags",
    "country_tags",
    "origin_tags"
]

for col in list_columns:
    if col in products_df.columns:
        products_df[f"{col}_parsed"] = products_df[col].apply(parse_list_column)

## 5. Normalize Text for Matching ##

In [204]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()

In [205]:
def normalize_tag_list(values):
    return [normalize_text(value) for value in values]

In [206]:
# Clean mapping keywords

label_mapping["label_keyword_clean"] = (
    label_mapping["label_keyword"]
    .astype(str)
    .str.strip()
    .str.lower()
)

confidence_mapping["label_keyword_clean"] = (
    confidence_mapping["label_keyword"]
    .astype(str)
    .str.strip()
    .str.lower()
)

## 6. Build Eco-Score Baseline from Mapping ##

In [207]:
eco_baseline_rows = label_mapping[
    (label_mapping["score_group"] == "eco_score")
    & (label_mapping["score_type"] == "baseline")
].copy()

In [208]:
def eco_grade_from_matched_label(value):
    value = normalize_text(value)
    if "eco-score" not in value:
        return None

    grade_text = value.replace("eco-score", "").strip()

    if grade_text in ["a-plus", "a+", "a plus"]:
        return "a-plus"
    if grade_text in ["a", "b", "c", "d", "e", "f"]:
        return grade_text
    return None

In [209]:
eco_baseline_rows = label_mapping[
    (label_mapping["score_group"] == "eco_score")
    & (label_mapping["score_type"] == "baseline")
].copy()

eco_baseline_rows["eco_grade"] = eco_baseline_rows["matched_label"].apply(
    eco_grade_from_matched_label
)

ECO_BASELINES = dict(
    zip(
        eco_baseline_rows["eco_grade"],
        eco_baseline_rows["environmental"]
    )
)

# Treat Eco-Score A+ the same as Eco-Score A.
if "a" in ECO_BASELINES:
    ECO_BASELINES["a-plus"] = ECO_BASELINES["a"]

ECO_BASELINES

{'a': 85, 'b': 70, 'c': 50, 'd': 30, 'e': 15, 'f': 5, 'a-plus': 85}

## 7. Prepare Score Columns ##

In [210]:
SCORE_COLUMNS = ["environmental", "social", "governance", "ethic"]

for col in SCORE_COLUMNS:
    label_mapping[col] = pd.to_numeric(label_mapping[col], errors="coerce").fillna(0)

confidence_mapping["confidence_weight"] = pd.to_numeric(
    confidence_mapping["confidence_weight"],
    errors="coerce"
).fillna(0)

## 8. Match Product Labels to Mapping ##

Source-aware matching:

- shared mappings apply to all products.
- source-specific mappings apply only to their database.


Use substring matching because product tags look like en:fair-trade, en:eu-organic, en:vegan.

In [211]:
def get_applicable_mapping_rows(source):
    database = SOURCE_TO_DATABASE.get(source)

    return label_mapping[
        (label_mapping["database"] == "shared")
        | (label_mapping["database"] == database)
    ].copy()

In [212]:
def normalize_match_text(value):
    value = normalize_text(value)
    value = value.replace("en:", "").replace("de:", "").replace("fr:", "")
    value = value.replace("-", " ").replace("_", " ").replace(":", " ")
    return " ".join(value.split())


def keyword_matches_tag(keyword, tag):
    keyword_clean = normalize_match_text(keyword)
    tag_clean = normalize_match_text(tag)

    if not keyword_clean:
        return False

    # Avoid tiny keywords like "nat" matching inside unrelated words.
    if len(keyword_clean) < 4:
        return keyword_clean in tag_clean.split()

    return keyword_clean in tag_clean

In [213]:
def match_labels(product_tags, source):
    product_tags_clean = normalize_tag_list(product_tags)
    applicable_mapping = get_applicable_mapping_rows(source)

    matched_rows = applicable_mapping[
        applicable_mapping["label_keyword_clean"].apply(
            lambda keyword: any(
                keyword_matches_tag(keyword, tag)
                for tag in product_tags_clean
            )
        )
    ].copy()

    return matched_rows

## 9. Avoid Double Counting ##

If multiple labels belong to the same score_group, keep only the strongest one.

Example: organic and EU Organic should not both stack.

In [214]:
def deduplicate_score_groups(matched_rows):
    if matched_rows.empty:
        return matched_rows

    matched_rows = matched_rows.copy()

    matched_rows["score_sum"] = (
        matched_rows["environmental"].abs()
        + matched_rows["social"].abs()
        + matched_rows["governance"].abs()
        + matched_rows["ethic"].abs()
    )

    return (
        matched_rows
        .sort_values("score_sum", ascending=False)
        .drop_duplicates(subset="score_group", keep="first")
    )

## 10. Confidence Calculation ##

In [215]:
# Use score_group to connect matched scoring rows with confidence rows

MIN_CONFIDENCE_SCORE = 15

def calculate_confidence(matched_rows, has_ecoscore):
    confidence = MIN_CONFIDENCE_SCORE

    if not matched_rows.empty:
        matched_score_groups = matched_rows["score_group"].dropna().unique()

        matched_confidence = confidence_mapping[
            confidence_mapping["score_group"].isin(matched_score_groups)
        ].copy()

        matched_confidence = matched_confidence.drop_duplicates(
            subset="score_group",
            keep="first"
        )

        confidence += matched_confidence["confidence_weight"].sum()

    if has_ecoscore:
        eco_confidence = confidence_mapping[
            confidence_mapping["score_group"] == "eco_score"
        ]["confidence_weight"]

        if not eco_confidence.empty:
            confidence += eco_confidence.iloc[0]

    return min(100, round(confidence, 1))

## 11. Core Scoring Function ##

In [216]:
NEUTRAL_BASELINE_SCORE = 50

OVERALL_WEIGHTS = {
    "environmental": 0.40,
    "social": 0.25,
    "governance": 0.15,
    "ethic": 0.20
}

In [217]:
def clamp(value, min_value=0, max_value=100):
    return max(min_value, min(max_value, value))

In [218]:
def score_product(row):
    source = row["source"]

    label_tags = row.get("label_tags_parsed", [])
    ecoscore_grade = normalize_text(row.get("ecoscore_grade_clean", ""))

    matched_rows = match_labels(label_tags, source)
    matched_rows = deduplicate_score_groups(matched_rows)

    environmental = NEUTRAL_BASELINE_SCORE
    social = NEUTRAL_BASELINE_SCORE
    governance = NEUTRAL_BASELINE_SCORE
    ethic = NEUTRAL_BASELINE_SCORE

    has_ecoscore = ecoscore_grade in ECO_BASELINES

    if has_ecoscore:
        environmental = ECO_BASELINES[ecoscore_grade]

    if not matched_rows.empty:
        environmental += matched_rows["environmental"].sum()
        social += matched_rows["social"].sum()
        governance += matched_rows["governance"].sum()
        ethic += matched_rows["ethic"].sum()

    environmental = clamp(environmental)
    social = clamp(social)
    governance = clamp(governance)
    ethic = clamp(ethic)

    overall = (
        environmental * OVERALL_WEIGHTS["environmental"]
        + social * OVERALL_WEIGHTS["social"]
        + governance * OVERALL_WEIGHTS["governance"]
        + ethic * OVERALL_WEIGHTS["ethic"]
    )

    confidence = calculate_confidence(matched_rows, has_ecoscore)

    explanation_notes = []
    if has_ecoscore:
        explanation_notes.append(f"Environmental baseline uses Eco-Score {ecoscore_grade}.")
    else:
        explanation_notes.append("No usable Eco-Score found; environmental score starts from the neutral baseline of 50.")

    if matched_rows.empty:
        explanation_notes.append("No positive or negative label evidence found in the current mapping.")
    else:
        explanation_notes.append(
            f"Matched {len(matched_rows)} scoring signal(s): "
            + ", ".join(matched_rows["matched_label"].astype(str).tolist())
        )

    return {
        "environmental_score": round(environmental, 1),
        "social_score": round(social, 1),
        "governance_score": round(governance, 1),
        "ethics_score": round(ethic, 1),
        "overall_score": round(overall, 1),
        "confidence_score": confidence,
        "matched_labels": matched_rows["matched_label"].tolist(),
        "matched_score_groups": matched_rows["score_group"].tolist(),
        "score_reasons": matched_rows["reason"].fillna("").tolist(),
        "score_explanation": " ".join(explanation_notes),
        "used_ecoscore": has_ecoscore,
        "ecoscore_grade_used": ecoscore_grade if has_ecoscore else None,
    }

## 12. Test One Product ##

In [219]:
sample_row = products_df[
    (products_df["source"] == "food")
    & (products_df["label_tags_parsed"].apply(len) > 0)
].iloc[0]

sample_result = score_product(sample_row)

sample_row[["product_name", "source", "label_tags", "ecoscore_grade_clean"]]

product_name                     Dunkle Schokolade mit ganzen Haselnüssen
source                                                               food
label_tags              ['en:fair-trade', 'en:fairtrade-international'...
ecoscore_grade_clean                                                    a
Name: 2, dtype: object

In [220]:
sample_result

{'environmental_score': np.int64(85),
 'social_score': np.int64(60),
 'governance_score': np.int64(50),
 'ethics_score': np.int64(50),
 'overall_score': np.float64(66.5),
 'confidence_score': np.int64(70),
 'matched_labels': ['Fairtrade International'],
 'matched_score_groups': ['fairtrade_certification'],
 'score_reasons': ['Certification promotes fair prices, better working conditions, and sustainable livelihoods for farmers and workers in global supply chains.'],
 'score_explanation': 'Environmental baseline uses Eco-Score a. Matched 1 scoring signal(s): Fairtrade International',
 'used_ecoscore': True,
 'ecoscore_grade_used': 'a'}

## 13. Score the Full Dataset ##

In [221]:
score_results = products_df.apply(score_product, axis=1)

In [222]:
score_results_df = pd.json_normalize(score_results)

In [223]:
scored_products_df = pd.concat(
    [products_df.reset_index(drop=True), score_results_df],
    axis=1
)

In [224]:
scored_products_df[[
    "source",
    "product_name",
    "brand",
    "environmental_score",
    "social_score",
    "governance_score",
    "ethics_score",
    "overall_score",
    "confidence_score",
    "matched_labels",
    "score_explanation"
]].head(50)

,source,product_name,brand,environmental_score,social_score,governance_score,ethics_score,overall_score,confidence_score,matched_labels,score_explanation
0,food,Toffifee 15er,Storck,50,50,50,50,50.0,15,[],No usable Eco-Score found; environmental score...
1,beauty,Bain Décalcifiant Réparateur,Kérastase,50,50,50,50,50.0,15,[],No usable Eco-Score found; environmental score...
2,food,Dunkle Schokolade mit ganzen Haselnüssen,fin CARRE,85,60,50,50,66.5,70,[Fairtrade International],Environmental baseline uses Eco-Score a. Match...
3,food,Hazelnut Milk Chocolate,fin CARRE,55,63,50,50,55.2,80,"[Fair for Life, UTZ Certified Cocoa]",Environmental baseline uses Eco-Score c. Match...
4,food,Toffifee weiße Schokolade,"Storck, Storck KG",50,50,50,50,50.0,15,[],No usable Eco-Score found; environmental score...
5,food,Dunkle Ganze Mandel,"Fin carré, Lidl",60,63,50,50,57.2,95,"[Fair for Life, UTZ Certified Cocoa, FSC]",Environmental baseline uses Eco-Score c. Match...
6,food,"Eat Natural Vegan - Erdnüsse, Kokos & dunkle S...","Eat Natural, Ferrero",55,50,50,63,54.6,85,"[Vegan, FSC, Vegetarian]",Environmental baseline uses Eco-Score c. Match...
7,food,Chocolate Fudge Brownie,Ben & Jerry's,70,50,50,55,59.0,60,[Vegetarian],Environmental baseline uses Eco-Score b. Match...
8,food,Milka Choc & Choc,Milka,30,50,50,50,42.0,55,[],Environmental baseline uses Eco-Score d. No po...
9,food,Magnum Mini Almond,MAGNUM,90,53,50,50,66.8,65,[Rainforest Alliance],Environmental baseline uses Eco-Score a-plus. ...


In [225]:
# Nutella products

nutella_df = scored_products_df[
    scored_products_df["product_name"].str.contains("nutella", case=False, na=False)
    | scored_products_df["brand"].str.contains("nutella|ferrero", case=False, na=False)
].copy()

nutella_df[[
    "source",
    "barcode",
    "product_name",
    "brand",
    "categories",
    "ecoscore_grade_clean",
    "environmental_score",
    "social_score",
    "governance_score",
    "ethics_score",
    "overall_score",
    "confidence_score",
    "matched_labels",
    "score_explanation"
]]

,source,barcode,product_name,brand,categories,ecoscore_grade_clean,environmental_score,social_score,governance_score,ethics_score,overall_score,confidence_score,matched_labels,score_explanation
6,food,8000500435618,"Eat Natural Vegan - Erdnüsse, Kokos & dunkle S...","Eat Natural, Ferrero","Chocolate cereal bars, Fruits cereal bars",c,55,50,50,63,54.6,85,"[Vegan, FSC, Vegetarian]",Environmental baseline uses Eco-Score c. Match...
15,food,8000500359884,Ferrero Rocher Haselnuss white,Ferrero Rocher,White chocolates with hazelnuts,f,5,50,50,50,32.0,55,[],Environmental baseline uses Eco-Score f. No po...
75,food,4008400159027,Ferrero Küsschen,Ferrero,"Chocolates with hazelnuts, Milk chocolate candies",unknown,50,50,50,50,50.0,15,[],No usable Eco-Score found; environmental score...


## 14. Basic Scoring Validation ##

In [226]:
# Check score ranges

scored_products_df[[
    "environmental_score",
    "social_score",
    "governance_score",
    "ethics_score",
    "overall_score",
    "confidence_score"
]].describe()

,environmental_score,social_score,governance_score,ethics_score,overall_score,confidence_score
count,442.000000,442.000000,442.000000,442.000000,442.000000,442.000000
mean,50.108597,51.235294,50.203620,52.803167,50.948869,42.036199
std,20.669343,2.808930,1.413952,5.365523,8.887080,26.891797
min,5.000000,50.000000,50.000000,50.000000,32.000000,15.000000
25%,50.000000,50.000000,50.000000,50.000000,50.000000,15.000000
50%,50.000000,50.000000,50.000000,50.000000,50.000000,30.000000
75%,55.000000,50.000000,50.000000,50.000000,52.800000,65.000000
max,100.000000,66.000000,60.000000,69.000000,72.600000,100.000000


In [227]:
# Check by source

scored_products_df.groupby("source")[[
    "environmental_score",
    "social_score",
    "governance_score",
    "ethics_score",
    "overall_score",
    "confidence_score"
]].mean().round(1)

,environmental_score,social_score,governance_score,ethics_score,overall_score,confidence_score
source,,,,,,
beauty,50.6,50.0,50.0,52.9,50.8,19.6
food,49.9,51.9,50.3,53.3,51.2,55.2
products,50.3,50.0,50.0,50.0,50.1,15.8


In [228]:
# Check matched label coverage

scored_products_df["matched_labels_count"] = scored_products_df["matched_labels"].apply(len)

scored_products_df.groupby("source")["matched_labels_count"].describe()

,count,mean,std,min,25%,50%,75%,max
source,,,,,,,,
beauty,99.0,0.484848,0.896292,0.0,0.0,0.0,0.0,3.0
food,285.0,1.150877,1.374455,0.0,0.0,1.0,2.0,7.0
products,58.0,0.051724,0.223404,0.0,0.0,0.0,0.0,1.0


In [ ]:
# Find products with no matched label-based signals

scored_products_df[
    scored_products_df["matched_labels_count"] == 0
][[
    "source",
    "product_name",
    "label_tags",
    "ecoscore_grade_clean",
    "overall_score",
    "confidence_score"
]].head(20)

,source,product_name,label_tags,ecoscore_grade_clean,overall_score,confidence_score
0,food,Toffifee 15er,[],unknown,50.0,15
1,beauty,Bain Décalcifiant Réparateur,[],unknown,50.0,15
4,food,Toffifee weiße Schokolade,"['en:green-dot', 'en:limited-edition']",unknown,50.0,15
8,food,Milka Choc & Choc,['en:green-dot'],d,42.0,55
13,food,Kinder Schokolade,"['en:no-gluten', 'en:no-preservatives', 'en:gr...",d,42.0,55
14,food,Peanut,"['en:green-dot', 'en:made-in-france']",e,36.0,55
15,food,Ferrero Rocher Haselnuss white,['fr:triman'],f,32.0,55
17,food,Pistazie Feinherb,['en:green-dot'],e,36.0,55
19,food,Puffreis mit Schokolade,"['fr:tidy-man', 'fr:triman']",e,36.0,55
20,food,Lindt Schokolade Goldhase,['fr:triman'],d,42.0,55


## 15. Save Scored Dataset ##

In [230]:
scored_products_df.to_csv(
    "../Processed/scored_products_germany_sample.csv",
    index=False
)

In [231]:
# Optional Streamlit-friendly version with fewer columns

streamlit_columns = [
    "source",
    "barcode",
    "product_name",
    "brand",
    "categories",
    "labels",
    "label_tags",
    "ecoscore_grade_clean",
    "environmental_score",
    "social_score",
    "governance_score",
    "ethics_score",
    "overall_score",
    "confidence_score",
    "matched_labels",
    "matched_score_groups",
    "score_reasons",
    "score_explanation",
    "image_url"
]

streamlit_scored_df = scored_products_df[
    [col for col in streamlit_columns if col in scored_products_df.columns]
].copy()

streamlit_scored_df.to_csv(
    "../Processed/streamlit_scored_products_germany_sample.csv",
    index=False
)